In [ ]:
!pip install -U langchain langchain-core langchain-community langchain-deepseek

In [28]:
import os
from langchain_deepseek import ChatDeepSeek
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser


In [38]:
llm = ChatDeepSeek(
    model="deepseek-chat", 
    temperature=0.4, 
    api_key="sk-b03ebd023fa744daa62e8d2c0d899111" 
)
# Initialize an empty conversation history.
chat_history = []

Student_Question_Prompt = ChatPromptTemplate.from_messages([
    ("system", (
        "You are an Elementary School Teaching Assistant. "
        "Explain things simply for kids in grades 1-5. "
        "Use analogies, provide numbered steps, and a final answer. "
        "Always suggest 3 similar practice questions at the end."
    )),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{question}"),
])

# 4. THE FUNCTION
def ask_teacher_question(student_question, history_list):
    # Prepare the chain
    chain = Student_Question_Prompt | llm | StrOutputParser()
    
    # Run the chain passing the history list
    response = chain.invoke({
        "question": student_question,
        "history": history_list
    })
    
    # Update the history list (Assignment 1 logic)
    history_list.append(HumanMessage(content=student_question))
    history_list.append(AIMessage(content=response))
    
    return response
